<a href="https://colab.research.google.com/github/MR-just01/Llama3.2-Reasoning/blob/main/notebooks/%2001_download_inspection%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import datasets
import transformers
import huggingface_hub
import peft
import accelerate


In [4]:
!pip install trl
import trl
print("trl:", trl.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 21.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


trl: 1.9.2


In [1]:
!pip install bitsandbytes
import bitsandbytes as bnb
print("bitsandbytes:", bnb.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 12.3 MB/s eta 0:00:00
bitsandbytes: 0.50.0


In [2]:
from datasets import load_dataset

gsm8k = load_dataset(
    "openai/gsm8k",
    "main"
)

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [ ]:
print(gsm8k)

In [3]:
print(gsm8k["train"].features)

{'question': Value('string'), 'answer': Value('string')}


In [4]:
sample = gsm8k["train"][0]
sample
from pprint import pprint
pprint(sample)

{'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\n'
           'Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and '
           'May.\n'
           '#### 72',
 'question': 'Natalia sold clips to 48 of her friends in April, and then she '
             'sold half as many clips in May. How many clips did Natalia sell '
             'altogether in April and May?'}


In [5]:
print("Train:", len(gsm8k["train"]))
print("Test :", len(gsm8k["test"]))

Train: 7473
Test : 1319


In [6]:
gsm_df = gsm8k["train"].to_pandas()

gsm_df.head()

,question,answer
0,Natalia sold clips to 48 of her friends in Apr...,Natalia sold 48/2 = <<48/2=24>>24 clips in May...
1,Weng earns $12 an hour for babysitting. Yester...,Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...
2,Betty is saving money for a new wallet which c...,"In the beginning, Betty has only 100 / 2 = $<<..."
3,"Julie is reading a 120-page book. Yesterday, s...",Maila read 12 x 2 = <<12*2=24>>24 pages today....
4,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...


In [7]:
gsm_df.info

<bound method DataFrame.info of                                                question  \
0     Natalia sold clips to 48 of her friends in Apr...   
1     Weng earns $12 an hour for babysitting. Yester...   
2     Betty is saving money for a new wallet which c...   
3     Julie is reading a 120-page book. Yesterday, s...   
4     James writes a 3-page letter to 2 different fr...   
...                                                 ...   
7468  Very early this morning, Elise left home in a ...   
7469  Josh is saving up for a box of cookies. To rai...   
7470  Colin can skip at six times the speed that Bra...   
7471  Janet, a third grade teacher, is picking up th...   
7472  At 30, Anika is 4/3 the age of Maddie. What wo...   

                                                 answer  
0     Natalia sold 48/2 = <<48/2=24>>24 clips in May...  
1     Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...  
2     In the beginning, Betty has only 100 / 2 = $<<...  
3     Maila read 12 x 2 = <<12*2=24>>24 pages today....  
4     He writes each friend 3*2=<<3*2=6>>6 pages a w...  
...                                                 ...  
7468  For the distance she traveled, Elise paid 23 -...  
7469  He makes $.5 profit on each bracelet because 1...  
7470  Tony can skip at twice the speed that Bruce ca...  
7471  Janet needs 35 lunches for the kids + 5 for th...  
7472  If Anika is 30 now, in 15 years, she'll be 30+...  

[7473 rows x 2 columns]>

In [8]:
# gsm_df.isnull().sum()
gsm_df.duplicated().sum()

np.int64(0)

In [9]:
gsm_df["question_length"] = gsm_df["question"].str.len()

gsm_df["question_length"].describe()

,question_length
count,7473.000000
mean,234.507427
std,93.904275
min,42.000000
25%,168.000000
50%,217.000000
75%,280.000000
max,985.000000


In [ ]:
gsm_df["answer_length"] = gsm_df["answer"].str.len()

gsm_df["answer_length"].describe()

In [ ]:
gsm_df.sample(5, random_state=42)

## GSM8K Observations

- Dataset Size: train  7473, test : 1319
- Columns: answer , question
- Missing Values: No missing values were found in either the question or answer columns.
- Duplicate Questions: No dulpicates were found .
- Average Question Length: appoximately 234.5 characters
- Average Answer Length: appoximately 287.49 characters
- Reasoning Included:yes it is included
- Final Answer Format: The final answer is appended at the end of the reasoning using the "####" delimiter.
- Potential Cleaning Required: Extract the final answer.
- Preserve reasoning steps.
- Remove calculation annotations (`<< >>`) if unnecessary.
- Normalize whitespace.
- Convert to the unified instruction format.

In [10]:
gsm_clean = gsm_df.copy()

In [11]:
gsm_clean["question"] = gsm_clean["question"].str.strip()

gsm_clean["answer"] = gsm_clean["answer"].str.strip()

In [12]:
import re

def extract_final_answer(answer):
    match = re.search(r"####\s*(.*)", answer)
    if match:
        return match.group(1).strip()
    return None

def extract_gsm8k_final_answer(answer):
    match = re.search(r"####\s*(.*)", answer)
    return match.group(1).strip() if match else None

In [14]:
gsm_clean[["answer", "final_answer"]].head()

KeyError: "['final_answer'] not in index"

In [15]:
gsm_clean["final_answer"].isnull().sum()

KeyError: 'final_answer'

StrategyQA

In [16]:
from datasets import load_dataset

strategyqa = load_dataset("ChilleD/StrategyQA")

README.md:   0%|          | 0.00/433 [00:00<?, ?B/s]

data/train-00000-of-00001-506370352f6228(…): reconstructing file:   0%|          |  0.00B /  369kB            

data/train-00000-of-00001-506370352f6228(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-bae602f3ee37f4c(…): reconstructing file:   0%|          |  0.00B /  161kB            

data/test-00000-of-00001-bae602f3ee37f4c(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1603 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/687 [00:00<?, ? examples/s]

In [17]:
print(strategyqa)

DatasetDict({
    train: Dataset({
        features: ['qid', 'term', 'description', 'question', 'answer', 'facts'],
        num_rows: 1603
    })
    test: Dataset({
        features: ['qid', 'term', 'description', 'question', 'answer', 'facts'],
        num_rows: 687
    })
})


In [18]:
print(strategyqa["train"].features)

{'qid': Value('string'), 'term': Value('string'), 'description': Value('string'), 'question': Value('string'), 'answer': Value('bool'), 'facts': Value('string')}


In [19]:
sample = strategyqa["train"][0]

from pprint import pprint
pprint(sample)

{'answer': False,
 'description': 'full contact combat sport',
 'facts': 'Mixed Martial arts in the UFC takes place in an enclosed structure '
          'called The Octagon. The Roman Colosseum games were fought in '
          'enclosed arenas where combatants would fight until the last man was '
          'standing. Mixed martial arts contests are stopped when one of the '
          'combatants is incapacitated. The Roman Colosseum was performed in '
          'front of crowds that numbered in the tens of thousands. Over 56,000 '
          'people attended UFC 193.',
 'qid': '4fd64bb6ce5b78ab20b6',
 'question': 'Is Mixed martial arts totally original from Roman Colosseum '
             'games?',
 'term': 'Mixed martial arts'}


In [20]:
import pandas as pd

strategy_df = strategyqa["train"].to_pandas()

strategy_df.head()

,qid,term,description,question,answer,facts
0,4fd64bb6ce5b78ab20b6,Mixed martial arts,full contact combat sport,Is Mixed martial arts totally original from Ro...,False,Mixed Martial arts in the UFC takes place in a...
1,f378f856bdaff39cdfa3,Cuisine of Hawaii,Cuisine of Hawaii,Is the cuisine of Hawaii suitable for a vegan?,False,"Per capita, Hawaiians are the second largest ..."
2,4e1b65e81ec09397b26e,Giant squid,Deep-ocean dwelling squid in the family Archit...,Is capturing giant squid in natural habitat im...,True,"Giant squids live between 1,000 and 3,800 feet..."
3,6d14da7484991bf588cf,Royal Air Force,Aerial warfare service branch of the British A...,Did the Royal Air Force fight in the Boxer Reb...,False,The Boxer Rebellion took place from 1899–1901 ...
4,3d01af5db202bc7d33b9,Eggplant,plant species Solanum melongena,Would someone in Mumbai refer to Solanum melon...,False,Mumbia is a city in India. India is a country ...


In [21]:
strategy_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1603 entries, 0 to 1602
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   qid          1603 non-null   object
 1   term         1603 non-null   object
 2   description  1603 non-null   object
 3   question     1603 non-null   object
 4   answer       1603 non-null   bool  
 5   facts        1603 non-null   object
dtypes: bool(1), object(5)
memory usage: 64.3+ KB


In [22]:
strategy_df.isnull().sum()

,0
qid,0
term,0
description,0
question,0
answer,0
facts,0


In [23]:
strategy_df.columns

Index(['qid', 'term', 'description', 'question', 'answer', 'facts'], dtype='object')

In [24]:
strategy_df["question"].duplicated().sum()

np.int64(0)

### Potential Cleaning Required

- Remove the `qid` column.
- Remove the `term` column.
- Remove the `description` column.
- Preserve the `question`.
- Preserve the supporting `facts`.
- Convert boolean labels (`True`/`False`) into textual labels (`Yes`/`No`).
- Normalize whitespace.
- Convert the dataset into the common instruction format.

In [26]:
strategy_clean = strategy_df.copy()

strategy_clean["answer"] = strategy_clean["answer"].map({
    True: "Yes",
    False: "No"
})

## **LogiQA**

In [29]:
from datasets import load_dataset

arc = load_dataset("allenai/ai2_arc", "ARC-Challenge")

README.md:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B /  190kB            

ARC-Challenge/train-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

ARC-Challenge/test-00000-of-00001.parque(…): reconstructing file:   0%|          |  0.00B /  204kB            

ARC-Challenge/test-00000-of-00001.parque(…): downloading bytes:           |  0.00B            

ARC-Challenge/validation-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 55.7kB            

ARC-Challenge/validation-00000-of-00001.(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

In [30]:
print(arc)

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 1119
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 1172
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 299
    })
})


In [31]:
print(arc["train"].features)

{'id': Value('string'), 'question': Value('string'), 'choices': {'text': List(Value('string')), 'label': List(Value('string'))}, 'answerKey': Value('string')}


In [32]:
from pprint import pprint

sample = arc["train"][0]

pprint(sample)

{'answerKey': 'A',
 'choices': {'label': ['A', 'B', 'C', 'D'],
             'text': ['dry palms',
                      'wet palms',
                      'palms covered with oil',
                      'palms covered with lotion']},
 'id': 'Mercury_SC_415702',
 'question': 'George wants to warm his hands quickly by rubbing them. Which '
             'skin surface will produce the most heat?'}


In [50]:
import pandas as pd

arc_df = arc["train"].to_pandas()

arc_df.head()

,id,question,choices,answerKey
0,Mercury_SC_415702,George wants to warm his hands quickly by rubb...,"{'text': ['dry palms', 'wet palms', 'palms cov...",A
1,MCAS_2009_5_6516,Which of the following statements best explain...,"{'text': ['The refrigerator door is smooth.', ...",B
2,Mercury_7233695,A fold observed in layers of sedimentary rock ...,"{'text': ['cooling of flowing magma.', 'conver...",B
3,Mercury_7041615,Which of these do scientists offer as the most...,"{'text': ['worldwide disease', 'global mountai...",D
4,Mercury_7041860,A boat is acted on by a river current flowing ...,"{'text': ['west', 'east', 'north', 'south'], '...",B


In [34]:
arc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1119 entries, 0 to 1118
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         1119 non-null   object
 1   question   1119 non-null   object
 2   choices    1119 non-null   object
 3   answerKey  1119 non-null   object
dtypes: object(4)
memory usage: 35.1+ KB


In [35]:
arc_df.isnull().sum()

,0
id,0
question,0
choices,0
answerKey,0


In [38]:
arc_df[["question"]].duplicated().sum()

np.int64(1)

In [39]:
arc_df.columns

Index(['id', 'question', 'choices', 'answerKey'], dtype='object')

In [40]:
arc_df.sample(5, random_state=42)

,id,question,choices,answerKey
243,Mercury_7030083,The day before the class is going to do a lab ...,"{'text': ['to prevent spills of chemicals', 't...",B
101,Mercury_SC_401589,Which mixture contains ingredients that can be...,"{'text': ['bread', 'fruit salad', 'ocean water...",B
961,Mercury_417154,"In 2005, a team of scientists discovered a pho...",{'text': ['Photosynthesis can occur without li...,B
1060,Mercury_SC_LBS10384,Vegetables can be scientifically classified by...,"{'text': ['size.', 'color.', 'shape of plant p...",D
522,TIMSS_2007_8_pg128,A sound is heard when you pluck a string on a ...,"{'text': ['The volume will stay the same, and ...",B


In [41]:
arc_df["question_length"] = arc_df["question"].str.len()

arc_df["question_length"].describe()

,question_length
count,1119.000000
mean,126.371761
std,80.042427
min,22.000000
25%,68.000000
50%,106.000000
75%,164.500000
max,683.000000


# ARC Challenge Observations

## Basic Information

- **Dataset Name:** ARC Challenge
- **Source:** Allen Institute for AI (AI2)
- **Hugging Face Repository:** `allenai/ai2_arc`
- **Reasoning Type:** Scientific and Logical Reasoning
- **Training Samples: 1119
- **Validation Samples:1172
- **Test Samples:299

---

## Structure

- **Columns:**
  - id
  - question
  - choices
  - answerKey

- **Missing Values: there are no missing values in the datasete
- **Duplicate Questions:1

## Text Statistics

- **Average Question Length:126.3 characters
- **Maximum Question Length:683 characters
- **Minimum Question Length: 22 characters

---

## Answer Format

- The dataset follows a **multiple-choice question (MCQ)** format.
- Each question contains **four answer options** labeled **A, B, C, and D**.
- The correct answer is stored in the **`answerKey`** column as the option label (e.g., A, B, C, or D).
- No step-by-step reasoning or explanation is provided.

---

## Data Quality

- **Missing Values: no missing values
- **Duplicate Questions
- **Formatting Issues:**
  - The `choices` column is stored as a nested dictionary containing option labels and option texts.
  - This nested structure cannot be directly processed by some Pandas operations such as `duplicated()`.

---

## Potential Cleaning Required

- Remove the `id` column since it is only metadata.
- Preserve the `question` column.
- Preserve all answer choices.
- Convert the nested `choices` dictionary into a structured text format or separate option columns.
- Convert the correct answer from its label (`A`, `B`, `C`, `D`) to the corresponding answer text.
- Normalize whitespace if required.
- Convert the dataset into the common instruction format used across all reasoning datasets.

---

## Remarks

The ARC Challenge dataset is a high-quality benchmark designed to evaluate scientific and logical reasoning abilities. Unlike GSM8K and StrategyQA, it does not provide reasoning steps or supporting facts. Instead, it evaluates the model's ability to select the correct answer from multiple choices based on scientific reasoning. This dataset adds diversity to the final training corpus by introducing multiple-choice scientific reasoning tasks.

In [43]:
duplicate_questions = arc_df[
    arc_df["question"].duplicated(keep=False)
]

duplicate_questions

,id,question,choices,answerKey,question_length
23,MEA_2011_8_8,How many times does Earth rotate on its axis i...,"{'text': ['once', 'twice', '24 times', '365 ti...",A,56
981,MEA_2012_5_8,How many times does Earth rotate on its axis i...,"{'text': ['once', 'twice', '24 times', '365 ti...",A,56


- Duplicate Questions:
  One duplicate question was identified during dataset inspection. The duplicated records will be reviewed during the data cleaning phase to determine whether they are exact duplicates before removal.

# **AQUA-RAT**












In [46]:
from datasets import load_dataset

aqua = load_dataset("deepmind/aqua_rat")

README.md:   0%|          | 0.00/5.89k [00:00<?, ?B/s]

raw/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 25.4MB            

raw/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

raw/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 74.0kB            

raw/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

raw/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 76.1kB            

raw/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/97467 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/254 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/254 [00:00<?, ? examples/s]

In [47]:
print(aqua)

DatasetDict({
    train: Dataset({
        features: ['question', 'options', 'rationale', 'correct'],
        num_rows: 97467
    })
    test: Dataset({
        features: ['question', 'options', 'rationale', 'correct'],
        num_rows: 254
    })
    validation: Dataset({
        features: ['question', 'options', 'rationale', 'correct'],
        num_rows: 254
    })
})


In [48]:
print(aqua["train"].features)

{'question': Value('string'), 'options': List(Value('string')), 'rationale': Value('string'), 'correct': Value('string')}


In [49]:
from pprint import pprint

sample = aqua["train"][0]

pprint(sample)

{'correct': 'E',
 'options': ['A)21', 'B)21.5', 'C)22', 'D)22.5', 'E)23'],
 'question': 'Two friends plan to walk along a 43-km trail, starting at '
             "opposite ends of the trail at the same time. If Friend P's rate "
             "is 15% faster than Friend Q's, how many kilometers will Friend P "
             'have walked when they pass each other?',
 'rationale': 'If Q complete x kilometers, then P completes 1.15x kilometers.\n'
              'x + 1.15x = 43\n'
              '2.15x=43\n'
              'x = 43/2.15 = 20\n'
              'Then P will have have walked 1.15*20=23 km.\n'
              'The answer is E.'}


In [51]:
import pandas as pd

aqua_df = aqua["train"].to_pandas()

aqua_df.head()

,question,options,rationale,correct
0,"Two friends plan to walk along a 43-km trail, ...","[A)21, B)21.5, C)22, D)22.5, E)23]","If Q complete x kilometers, then P completes 1...",E
1,"In the coordinate plane, points (x, 1) and (5,...","[A)4 and 1, B)1 and 5, C)5 and 1, D)3 and 5, E...",Line k passes through the origin and has slope...,C
2,"For all numbers p and q, the operation @ is de...","[A)II, B)I and II, C)I and III, D)II and III, ...",p@q = p^2 - pq=p(p-q).... so p@q will be zero ...,B
3,Carl is facing very difficult financial times ...,"[A)$1600, B)$2000, C)$2150, D)$2500, E)$12000]","Usually, you are given the annual rate of inte...",A
4,The speed at which a man can row a boat in sti...,"[A)18 seconds, B)27 seconds, C)26 seconds, D)1...",Speed of the boat downstream = 25 +11\n= 36 km...,E


In [52]:
aqua_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97467 entries, 0 to 97466
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   question   97467 non-null  object
 1   options    97467 non-null  object
 2   rationale  97467 non-null  object
 3   correct    97467 non-null  object
dtypes: object(4)
memory usage: 3.0+ MB


In [53]:
aqua_df.isnull().sum()

,0
question,0
options,0
rationale,0
correct,0


In [55]:
aqua_df["question"].duplicated().sum()

np.int64(16543)

In [56]:
aqua_df["question_length"] = aqua_df["question"].str.len()

aqua_df["question_length"].describe()

,question_length
count,97467.000000
mean,163.088286
std,90.070565
min,3.000000
25%,101.000000
50%,147.000000
75%,206.000000
max,1137.000000


In [57]:
aqua_df.sample(5, random_state=42)

,question,options,rationale,correct,question_length
58044,"Complete the below series\n5,6,7,8,10,11,14, ?","[A)12, B)13, C)14, D)15, E)16]",when we separate odd and even places the we ge...,D,45
8745,Two numbers are in the ratio of 5:9. If 25 be ...,"[A)150,170, B)150,270, C)50,270, D)180,270, E)...","(5x-25):(9x-25) = 35:59\nx = 30 => 150,270\nAN...",B,117
32814,Peter invests a sum of money and gets back an ...,"[A)653, B)664, C)698, D)744, E)700]",Since both Peter and David invested the same a...,A,255
33295,5670/(28*13.5) = ?,"[A)11, B)15, C)16, D)19, E)18]",B\n15\n? = 5670/378 = 15,B,18
60034,If the compound interest on a certain sum for ...,"[A)Rs.1500, B)Rs.1450, C)Rs.1550, D)Rs.1650, E...",Let principal be P.\nThen Amount = P + 1590\nA...,A,142


In [59]:
duplicate_questions["question"].value_counts().head(20)

,count
question,
"A train passes a station platform in 36 seconds and a man standing on the platform in 20 seconds. If the speed of the train is 54 km/hr, what is the length of the platform?",52
A train running at the speed of 60 km/hr crosses a pole in 9 seconds. What is the length of the train?,44
Find the value of y from (12)^3 x 6^4 ÷ 432 = y?,26
A train passes a station platform in 36 sec and a man standing on the platform in 20 sec. If the speed of the train is 54 km/hr. What is the length of the platform?,25
A man can row his boat with the stream at 6 km/h and against the stream in 4 km/h. The man's rate is?,25
A 300 m long train crosses a platform in 39 sec while it crosses a signal pole in 18 sec. What is the length of the platform?,25
"A train covers a distance of 12 km in 10 min. If it takes 6 sec to pass a telegraph post, then the length of the train is?",24
"The two trains of lengths 400 m, 600 m respectively, running at same directions. The faster train can cross the slower train in 180 sec, the speed of the slower train is 48 km. then find the speed of the faster train?",24
"A man can row with a speed of 15 kmph in still water. If the stream flows at 5 kmph, then the speed in downstream is?",24


In [60]:
q = duplicate_questions["question"].value_counts().index[0]

duplicate_questions[
    duplicate_questions["question"] == q
]

,question,options,rationale,correct,question_length
610,A train passes a station platform in 36 second...,"[A)378, B)240, C)772, D)281, E)213]",Speed = (54 * 5/18) m/sec = 15 m/sec. Length o...,B,172
1697,A train passes a station platform in 36 second...,"[A)388, B)240, C)88, D)66, E)221]",Speed = (54 * 5/18) m/sec = 15 m/sec. Length o...,B,172
4385,A train passes a station platform in 36 second...,"[A)388, B)378, C)240, D)388, E)771]",Speed = [54 * 5/18] m/sec = 15 m/sec.\nLength ...,C,172
4737,A train passes a station platform in 36 second...,"[A)338, B)240, C)287, D)267, E)191]",Speed = (54 * 5/18) m/sec = 15 m/sec. Length o...,B,172
7036,A train passes a station platform in 36 second...,"[A)180 m, B)200 m, C)240 m, D)320 m, E)None]",Sol.\nSpeed = [54 * 5/18] m/sec = 15 m/sec.\nL...,C,172
9066,A train passes a station platform in 36 second...,"[A)816 m, B)577 m, C)240 m, D)176 m, E)126 m]",Speed = [54 * 5/18] m/sec = 15 m/sec.\nLength ...,C,172
9079,A train passes a station platform in 36 second...,"[A)210, B)220, C)240, D)250, E)260]",Speed = [54 * 5/18] m/sec = 15 m/sec.\nLength ...,C,172
12685,A train passes a station platform in 36 second...,"[A)288, B)240, C)277, D)127, E)922]",Speed = (54 * 5/18) m/sec = 15 m/sec. Length o...,B,172
13749,A train passes a station platform in 36 second...,"[A)120 m, B)240 m, C)300 m, D)200 m, E)None of...",Explanation:\nSpeed = (54 * 5/18) m/sec = 15 m...,B,172
13914,A train passes a station platform in 36 second...,"[A)2387, B)209, C)240, D)278, E)121]",Speed = [54 * 5/18] m/sec = 15 m/sec.\nLength ...,C,172
